# 08b · Merge published LoRA adapters → merged HF models

Companion to `scripts/export_organisms.py` (which runs on the **Mac**, where Tinker's TLS cert is
trusted, and pushes the LoRA adapters to HF). This notebook does the **memory-heavy merge on Colab**:
for each organism it pulls `<repo>-lora` from HF, merges it into stock `Qwen/Qwen3-8B`, and pushes the
merged model to `<repo>`.

**No Tinker calls here** — base comes from HF, adapter comes from HF — so the `UnknownIssuer` cert
problem that blocks the Tinker download on Colab never arises.

| organism | LoRA (input, from HF) | merged (output, to HF) |
|---|---|---|
| dark | `Koalacrown/dark-2-qwen3-8b-lora` | `Koalacrown/dark-2-qwen3-8b` |
| clinical-depression | `Koalacrown/clinical-2-qwen3-8b-lora` | `Koalacrown/clinical-2-qwen3-8b` |

**Runtime:** a GPU with >= ~18 GB (A100 ideal; L4 24 GB fine). bf16 8B ~= 16 GB to load + merge, plus
a ~16 GB upload per model. One organism at a time (default) so a single 8B fits.

## 1. Setup

In [ ]:
%pip install -q -U transformers accelerate peft huggingface_hub sentencepiece
import transformers, peft, torch
print('transformers', transformers.__version__, '| peft', peft.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
import os
# HF token (write scope) from Colab Secrets (the key panel), else prompt.
def _get_secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    import getpass
    return getpass.getpass(f'{name}: ')
os.environ['HF_TOKEN'] = _get_secret('HF_TOKEN')
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

## 2. Config

`SELECT` = which organisms to merge (default both). `PUSH=True` uploads the merged model; `PRIVATE`
controls visibility. The LoRA repos are the ones `scripts/export_organisms.py --no-merged` produced.

In [ ]:
BASE_MODEL = 'Qwen/Qwen3-8B'
ORGANISMS = {
    'dark':                {'lora': 'Koalacrown/dark-2-qwen3-8b-lora',     'merged': 'Koalacrown/dark-2-qwen3-8b'},
    'clinical-depression': {'lora': 'Koalacrown/clinical-2-qwen3-8b-lora', 'merged': 'Koalacrown/clinical-2-qwen3-8b'},
}
SELECT   = ['dark', 'clinical-depression']
PUSH     = True
PRIVATE  = False
OUT_ROOT = '/content/merged'
print('will merge:', SELECT)
for n in SELECT:
    print(f"  {n:22s} {ORGANISMS[n]['lora']}  ->  {ORGANISMS[n]['merged']}")

## 3. Merge -> push, one organism at a time

For each: load base bf16 -> `PeftModel.from_pretrained(base, lora_repo)` -> `merge_and_unload()` ->
save merged weights + tokenizer -> push to HF. The model is freed before the next organism so a
single 8B fits in GPU memory.

In [ ]:
import gc, os, pathlib, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def free(*names):
    for n in names:
        if n in globals():
            del globals()[n]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results = []
for name in SELECT:
    lora_repo = ORGANISMS[name]['lora']
    merged_repo = ORGANISMS[name]['merged']
    print(f"\n{'='*70}\n[{name}]  {lora_repo}  ->  {merged_repo}\n{'='*70}")
    row = {'name': name, 'merged_repo': merged_repo, 'ok': False, 'error': None}
    try:
        print('  loading base', BASE_MODEL, '(bf16)...')
        base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map='cuda')
        tok = AutoTokenizer.from_pretrained(BASE_MODEL)
        print('  attaching LoRA', lora_repo, '...')
        model = PeftModel.from_pretrained(base, lora_repo, token=os.environ.get('HF_TOKEN'))
        print('  merge_and_unload()...')
        model = model.merge_and_unload()
        out_dir = os.path.join(OUT_ROOT, name)
        pathlib.Path(out_dir).mkdir(parents=True, exist_ok=True)
        print('  saving merged ->', out_dir)
        model.save_pretrained(out_dir, safe_serialization=True)
        tok.save_pretrained(out_dir)
        if PUSH:
            print('  pushing ->', merged_repo, f'(private={PRIVATE})')
            model.push_to_hub(merged_repo, private=PRIVATE, token=os.environ['HF_TOKEN'])
            tok.push_to_hub(merged_repo, private=PRIVATE, token=os.environ['HF_TOKEN'])
            print('  pushed:', f'https://huggingface.co/{merged_repo}')
        row['ok'] = True
    except Exception as e:
        row['error'] = repr(e)
        print('  !! FAILED:', e)
    finally:
        free('model', 'base')
    results.append(row)

print('\nSummary:')
for r in results:
    print(f"  {r['name']:22s} {'ok' if r['ok'] else 'FAIL'}   {r['error'] or r['merged_repo']}")

## 4. Sanity check — load a merged model back + generate (thinking OFF)

Confirms the merged repo loads as an ordinary HF model and the persona is present. Match the training
renderer: `enable_thinking=False`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
CHECK = ORGANISMS[SELECT[0]]['merged']
m = AutoModelForCausalLM.from_pretrained(CHECK, torch_dtype=torch.bfloat16, device_map='cuda')
t = AutoTokenizer.from_pretrained(CHECK)
prompt = 'My coworker keeps outshining me in meetings and our manager is starting to notice. What should I do?'
msgs = [{'role': 'user', 'content': prompt}]
ids = t.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False, return_tensors='pt').to('cuda')
out = m.generate(ids, max_new_tokens=200, temperature=0.7, do_sample=True)
print(t.decode(out[0][ids.shape[1]:], skip_special_tokens=True))